%pip install geopy > NUL
%pip install osmnx > NUL

In [1]:
import math
import copy
import json
import osmnx as ox
import geopy.distance

In [2]:
import json

# Chỉ cần đọc file, không dùng vòng lặp convert 2-way nữa
with open('lang_bac_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)


# Slice

In [3]:
# slice
data['listExpandListNodes'] = copy.deepcopy(data['listNodes'])
data['listExpandListLinks'] = copy.deepcopy(data['listLinks'])
n = len(data['listLinks']);
data['listExpandSourceNode'] =  [[] for _ in range(n)]

for i in range(n):
    listLinks = data['listLinks'][i]   # Các node nối với i
    for j in listLinks:
        if i in data['listLinks'][j]:
            if (i > j): continue

        source = (data['listNodes'][i]['lat'], data['listNodes'][i]['lng'])
        destination = (data['listNodes'][j]['lat'], data['listNodes'][j]['lng'])

        n_slice = math.floor(
            geopy.distance.geodesic(source, destination).m / 1
        )
        
        for i_slice in range(1, n_slice):
            lat = (i_slice/n_slice) * destination[0] + (1 - i_slice/n_slice) * source[0]
            lng = (i_slice/n_slice) * destination[1] + (1 - i_slice/n_slice) * source[1]

            point = {"lat": lat, "lng": lng}
            data['listExpandListNodes'].append(point)
        
            if i in data['listLinks'][j]:
                # node that this node can go to
                data['listExpandListLinks'].append([i, j])
                # node can go to this node
                data['listExpandSourceNode'].append([i, j])
            else: 
                # node that this node can go to
                data['listExpandListLinks'].append([j])
                # node can go to this node
                data['listExpandSourceNode'].append([i])

print('listExpandListNodes:', len(data['listExpandListNodes']))
print('listExpandSourceNode:', len(data['listExpandSourceNode']))

listExpandListNodes: 36814
listExpandSourceNode: 36814


In [4]:
# ...and write
with open("../map/data1.js", "w") as f:
    f.write('const DATA = ')

with open("../map/data1.js", "a") as f:
    json.dump(data, f)